# Seller churn prediction: rolling backtest

Which active sellers are likely to stop selling in the next 90 days, and how much of the sales at risk could a limited outreach budget reach?

**Before running:** put the Olist CSV files in `../data/raw/` (see `../data/README.md`), then run the cells in order. Files are read from `DATA_DIR`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, "../python")
import churn_pipeline as cp

DATA_DIR = Path("../data/raw")
OUT_DIR = Path("../outputs")
pd.set_option("display.float_format", "{:,.3f}".format)

## 1. Load orders by seller
One row per order and seller, with sales, delivery lateness, review score and category.

In [ ]:
so = cp.load_seller_orders(DATA_DIR)
sellers = cp.load_sellers(DATA_DIR)
print(so.shape, so["seller_id"].nunique(), "sellers")
so.head()

## 2. Explore (your decisions start here)
Check the order volume and the gaps between orders before trusting the 90-day churn window.

In [ ]:
monthly = so.set_index("ts").resample("MS").agg(orders=("order_id", "nunique"), sellers=("seller_id", "nunique"), gmv=("gmv", "sum"))
monthly

In [ ]:
# How long do sellers go quiet between orders? If most gaps are far below 90 days, a 90-day window is a clear signal.
gaps = (so.sort_values("ts").groupby("seller_id")["ts"].diff().dt.days.dropna())
gaps.describe(percentiles=[.5, .75, .9, .95, .99])

## 3. Build the seller-month table
For every month start from Apr 2017, one row per **active** seller (1+ order in the last 90 days). Features use only orders before the date; `churned = 1` means no orders in the next 90 days.

In [ ]:
panel = cp.build_panel(so, sellers)
print(panel.shape, "rows;", panel["cutoff"].nunique(), "cutoff months")
panel.groupby("cutoff").agg(active_sellers=("seller_id", "count"), churn_rate=("churned", "mean"))

## 4. Rolling backtest
For each test month the models train only on earlier months whose 90-day outcome was already known on that date. Two simple rules are scored the same way as a benchmark: the longer a seller has been silent, and the biggest recent sellers.

In [ ]:
bt = cp.rolling_backtest(panel, top_k=cp.TOP_K)
summary = cp.summarize(bt)
summary

## 5. Results in money
`share_reached` = share of the sales at risk (sales of sellers who really churned, last 90 days before the date) that sit with the top-K sellers the model flags each month.

In [ ]:
cp.make_charts(bt, summary, OUT_DIR)
bt.pivot(index="test_cutoff", columns="model", values="share_reached")

## 6. What drives churn?

In [ ]:
imp = cp.feature_importance(panel)
imp.head(10)

## 7. Save outputs

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
bt.to_csv(OUT_DIR / "backtest_by_month.csv", index=False)
summary.to_csv(OUT_DIR / "backtest_summary.csv", index=False)
imp.to_csv(OUT_DIR / "feature_importance.csv", index=False)